# 07 — VCOD completeness audit and paired report
Run after every declared final seed has all 12 primary cells. This notebook inventories Drive artifacts, rejects incomplete or duplicate cells, invokes the paired source-video summarizer, and displays the generated report. It never reads tuning or smoke runs.

In [ ]:
# Fresh-kernel bootstrap. A GPU runtime is used only because the shared repository bootstrap verifies model assets.
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/cod-ssl'
DECLARED_SEEDS = [42, 43, 44]

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys
project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks]'], check=True)
bootstrap_env = os.environ.copy()
dino_weights = Path(DRIVE_ROOT) / 'checkpoints/dinov3_vitb16.pth'
if not dino_weights.is_file():
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
subprocess.run([sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
                '--project-dir', str(project_dir), '--drive-root', DRIVE_ROOT,
                '--state-file', str(state_file)],
               cwd=project_dir, env=bootstrap_env, check=True)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = Path(state['project_dir'])
VCOD_ROOT = Path(state['drive_root']) / 'vcod'
RUNS_ROOT = VCOD_ROOT / 'runs'
REPORT_ROOT = VCOD_ROOT / 'reports/primary_comparison'
os.chdir(PROJECT_DIR)
print('Reading final runs only from', RUNS_ROOT)

In [ ]:
# Inventory the declared matrix before bootstrapping statistics.
import pandas as pd
from IPython.display import display
expected = {(dataset, regime, system, seed)
            for dataset, regime in [('moca_mask', 'default'),
                                    ('camovid60k', 'small_displacement'),
                                    ('camovid60k', 'large_displacement')]
            for system in ('DS', 'VI', 'DT', 'VV') for seed in DECLARED_SEEDS}
found, inventory = {}, []
for summary_path in RUNS_ROOT.rglob('summary.json'):
    summary = json.loads(summary_path.read_text())
    run = summary['run']
    key = (run['dataset'], run.get('regime') or 'default', run['system_id'], int(run['seed']))
    if key in found: raise ValueError(f'Duplicate final cell {key}: {found[key]} and {summary_path}')
    found[key] = summary_path
    inventory.append({'dataset': key[0], 'regime': key[1], 'system': key[2],
                      'seed': key[3], 'summary': str(summary_path),
                      'evaluation_complete': (summary_path.parent / 'EVALUATION_COMPLETE').is_file()})
missing, unexpected = expected - found.keys(), found.keys() - expected
display(pd.DataFrame(inventory).sort_values(['dataset', 'regime', 'seed', 'system']))
if unexpected: raise ValueError(f'Unexpected cells in final root: {sorted(unexpected)}')
if missing: raise ValueError(f'Missing {len(missing)} primary cells: {sorted(missing)}')
if not all(row['evaluation_complete'] for row in inventory):
    raise ValueError('At least one cell lacks EVALUATION_COMPLETE.')
print(f'Complete: {len(found)} primary cells across {len(DECLARED_SEEDS)} seeds.')

In [ ]:
# Generate Markdown, CSV, and JSON only from saved run artifacts.
subprocess.run([sys.executable, 'scripts/summarize_results.py',
                '--runs-root', str(RUNS_ROOT),
                '--matrix', 'configs/experiments/vcod_primary_2x2.yaml',
                '--output', str(REPORT_ROOT)], cwd=PROJECT_DIR, check=True)
if hasattr(os, 'sync'): os.sync()

In [ ]:
# Display the generated primary tables and report; Drive retains the authoritative artifacts.
from IPython.display import Markdown, display
display(pd.read_csv(REPORT_ROOT / 'primary_results.csv'))
display(pd.read_csv(REPORT_ROOT / 'paired_comparisons.csv'))
display(pd.read_csv(REPORT_ROOT / 'motion_interactions.csv'))
display(Markdown((REPORT_ROOT / 'report.md').read_text()))
print('Final report:', REPORT_ROOT)

Diagnostics, qualitative clips, and conditional follow-ups remain separate from the locked primary matrix. Do not copy tuning or smoke summaries into `vcod/runs`.